In [11]:
# -*- coding: utf-8 -*-
"""
Rollout JSONL -> per-image CSV extractor (sample every k files by order).

What this script does:
1) Find all `*.jsonl` under the given `rollout` folder and sort by the numeric `step` in the filename.
2) Pick every k-th file by **order index** (1-based), e.g., the 20th, 40th, 60th files when k=20.
3) For each sampled step file, read JSONL lines, and for each line:
   - extract image_index from `input` text,
   - extract crop coordinates from <tool_call> JSON in `output`,
   - extract reasoning from <think>...</think> in `output`,
   - extract final answer from \boxed{...} in `output`,
   - extract score from record["score"],
   - extract disease name from `input` text.
4) Group rows by image_index and write one CSV per image:
   verl_new/generations/.../rollout_crop_analysis/step_{step}/image_{image_index}_text_analysis.csv
   
The code is defensive to formatting quirks and will skip lines that cannot be parsed.
"""

import os
import re
import json
import csv
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

# ------------ Configuration -------------
K = 20  # sample every K files by order (1-based indexing: 20th, 40th, ...)
ROLLOUT_DIR = Path("/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout")
OUTPUT_ROOT = Path("/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis")

# Make sure output root exists
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ------------ Regex helpers -------------
RE_IMAGE_INDEX = re.compile(r"repeat the image index\s*([\-]?\d+)", re.IGNORECASE)
RE_TOOL_BLOCK = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL | re.IGNORECASE)
RE_THINK = re.compile(r"<think>(.*?)</think>", re.DOTALL | re.IGNORECASE)
# Accept either \boxed{...} or \\boxed{...} in the raw string
RE_BOXED = re.compile(r"(?:\\boxed|\\\\boxed)\{(.*?)\}", re.DOTALL)
# 1) Does this patient have XYZ?  2) does XYZ exists in this patient
RE_DISEASE_1 = re.compile(r"Does this patient have\s+([A-Za-z][A-Za-z \-]+?)\s*\?", re.IGNORECASE)
RE_DISEASE_2 = re.compile(r"does\s+([A-Za-z][A-Za-z \-]+?)\s+exists?\s+in\s+this\s+patient", re.IGNORECASE)

PLACEHOLDER_REASONS = {"your reason", "'your reason'", '"your reason"'}
PLACEHOLDER_ANSWERS = {"your answer", "'your answer'", '"your answer"'}

def extract_step_from_filename(p: Path) -> Optional[int]:
    """
    Filename is like '48.jsonl' or '2.jsonl' -> return 48 or 2.
    """
    try:
        return int(p.stem)
    except Exception:
        return None

def extract_image_index_from_input(input_text: str) -> Optional[int]:
    m = RE_IMAGE_INDEX.search(input_text or "")
    if m != "-1":
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None

def extract_disease_from_input(input_text: str) -> Optional[str]:
    if not input_text:
        return None
    m1 = RE_DISEASE_1.search(input_text)
    if m1:
        return m1.group(1).strip()
    m2 = RE_DISEASE_2.search(input_text)
    if m2:
        return m2.group(1).strip()
    # fallback: sometimes appears like "disease: Atelectasis"
    m3 = re.search(r"disease\s*:\s*([A-Za-z][A-Za-z \-]+)", input_text, flags=re.IGNORECASE)
    if m3:
        return m3.group(1).strip()
    return None

def extract_tool_coords_from_output(output_text: str) -> Optional[Tuple[Optional[int], Optional[List[int]]]]:
    """
    Return (index, coordinates_list) if found, else None.
    """
    if not output_text:
        return None
    m = RE_TOOL_BLOCK.search(output_text)
    if not m:
        return None
    json_blob = m.group(1)
    # Try to JSON parse as-is; if it fails due to single quotes etc, try a mild normalization
    try:
        tool_obj = json.loads(json_blob)
    except Exception:
        # Normalize single quotes to double quotes carefully where it looks like JSON keys/strings
        sanitized = re.sub(r"'", '"', json_blob)
        # Remove trailing commas if any
        sanitized = re.sub(r",\s*([}\]])", r"\1", sanitized)
        try:
            tool_obj = json.loads(sanitized)
        except Exception:
            return None
    
    if isinstance(tool_obj, dict):
        args = tool_obj.get("arguments", {})
        if isinstance(args, dict):
            idx = args.get("index", None)
            coords = args.get("coordinates", None)
            if isinstance(coords, list) and all(isinstance(v, (int, float)) for v in coords):
                # cast floats to int if any
                coords = [int(round(v)) for v in coords]
            return (idx, coords)
    return None

def _strip_quotes_lower(s: str) -> str:
    """Normalize by stripping surrounding single/double quotes and lowering."""
    if s is None:
        return ""
    s = s.strip()
    # Strip one layer of matching quotes
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()
    return s.lower()

def extract_think(output_text: str) -> Optional[str]:
    """Return the first non-placeholder <think> ... </think> match.
    If the first match is a placeholder like 'your reason', try the next one.
    """
    if not output_text:
        return None
    matches = list(RE_THINK.finditer(output_text))
    if not matches:
        return None
    for m in matches:
        cand = (m.group(1) or "").strip()
        norm = _strip_quotes_lower(cand)
        if norm not in PLACEHOLDER_REASONS:
            return cand
    # If all are placeholders, return the last one (still informative for debugging)
    return (matches[-1].group(1) or "").strip()

def extract_boxed_answer(output_text: str) -> Optional[str]:
    """Return the first non-placeholder \\boxed{...} match.
    If the first match is a placeholder like 'your answer', try the next.
    """
    if not output_text:
        return None
    matches = list(RE_BOXED.finditer(output_text))
    if not matches:
        # also try raw single backslash pattern
        matches = list(re.finditer(r"\\boxed\{(.*?)\}", output_text, re.DOTALL))
        if not matches:
            return None
    for m in matches:
        cand = (m.group(1) or "").strip()
        norm = _strip_quotes_lower(cand)
        if norm not in PLACEHOLDER_ANSWERS:
            return cand
    return (matches[-1].group(1) or "").strip()

def parse_jsonl_line(line: str) -> Optional[Dict[str, Any]]:
    """
    Try to load a JSONL record. Return dict or None.
    """
    line = line.strip()
    if not line:
        return None
    try:
        obj = json.loads(line)
        return obj
    except Exception:
        # Some logs might have trailing commas or junk, attempt a mild cleanup
        try:
            # Remove stray control characters that often sneak in
            cleaned = line.replace("\r", " ").replace("\n", " ")
            # Remove superfluous commas before closing braces/brackets
            cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
            obj = json.loads(cleaned)
            return obj
        except Exception:
            return None

def process_step_file(step_path: Path, output_root: Path) -> None:
    step_num = extract_step_from_filename(step_path)
    if step_num is None:
        return
    
    # Collect rows per image_index
    rows_by_image: Dict[int, List[Dict[str, Any]]] = {}
    
    with step_path.open("r", encoding="utf-8") as f:
        for ln, raw in enumerate(f, start=1):
            rec = parse_jsonl_line(raw)
            if not rec:
                continue
            input_text = rec.get("input", "") or ""
            output_text = rec.get("output", "") or ""
            score = rec.get("score", None)
            
            image_idx = extract_image_index_from_input(input_text)
            if image_idx is None:
                # If image index is missing in input, try read from tool_call block
                tc = extract_tool_coords_from_output(output_text)
                if tc and isinstance(tc[0], int):
                    image_idx = int(tc[0])
            
            disease = extract_disease_from_input(input_text) or ""
            
            # Parse tool call for coordinates
            coords = None
            tool_info = extract_tool_coords_from_output(output_text)
            if tool_info:
                _, coords = tool_info
            
            thinking = extract_think(output_text) or ""
            answer = extract_boxed_answer(output_text) or ""
            
            if image_idx is None:
                # Skip if we cannot anchor this record to an image
                continue
            
            row = {
                "image_index": image_idx,
                "crop_coordinates": coords if coords is not None else "",
                "reasoning": thinking,
                "answer": answer,
                "score": score,
                "disease": disease,
            }
            rows_by_image.setdefault(image_idx, []).append(row)
    
    # Write one CSV per image
    step_out_dir = output_root / f"step_{step_num}"
    step_out_dir.mkdir(parents=True, exist_ok=True)
    
    headers = ["image_index", "crop_coordinates", "reasoning", "answer", "score", "disease"]
    for img_idx, rows in rows_by_image.items():
        out_csv = step_out_dir / f"image_{img_idx}_text_analysis.csv"
        with out_csv.open("w", newline="", encoding="utf-8") as wf:
            writer = csv.DictWriter(wf, fieldnames=headers)
            writer.writeheader()
            for r in rows:
                # stringify coordinates for CSV readability
                r2 = dict(r)
                if isinstance(r2.get("crop_coordinates"), list):
                    r2["crop_coordinates"] = json.dumps(r2["crop_coordinates"], ensure_ascii=False)
                writer.writerow(r2)

def main(rollout_dir: Path, output_root: Path, k: int = 20) -> None:
    if not rollout_dir.exists():
        print(f"[Warning] Rollout directory not found: {rollout_dir.resolve()}")
        print("No files were processed. Please ensure the path is correct.")
        return
    
    files = sorted(
        [p for p in rollout_dir.glob("*.jsonl") if extract_step_from_filename(p) is not None],
        key=lambda p: extract_step_from_filename(p)
    )
    if not files:
        print(f"[Info] No '*.jsonl' files under: {rollout_dir}")
        return
    
    print(f"[Info] Found {len(files)} jsonl files.")
    
    sampled: List[Path] = []
    # 1-based sampling: pick the 20th, 40th, 60th, ...
    for idx, p in enumerate(files, start=1):
        if idx % k == 0:
            sampled.append(p)
    print(f"[Info] Sampling every {k}th file by order (1-based) -> {len(sampled)} files selected.")
    if not sampled:
        print("[Info] No files selected by the sampling rule.")
        return
    
    for sp in sampled:
        step_num = extract_step_from_filename(sp)
        print(f"  - Processing step file: {sp.name} (step={step_num})")
        try:
            process_step_file(sp, output_root)
        except Exception as e:
            print(f"[Error] Failed processing {sp}: {e}")
    
    print("[Done] Processing complete.")
    print(f"Outputs are (per image) under: {output_root}/step_{{step}}/image_{{image_index}}_text_analysis.csv")



main(ROLLOUT_DIR, OUTPUT_ROOT, K)


# # Save this script to disk so you can reuse it locally
# SCRIPT_PATH = Path("/mnt/data/rollout_text_analysis.py")
# with SCRIPT_PATH.open("w", encoding="utf-8") as sf:
#     sf.write(open(__file__, "r", encoding="utf-8").read())

# # Try to run once (will just warn if the path doesn't exist in this environment)
# main(ROLLOUT_DIR, OUTPUT_ROOT, K)

# print(f"\nScript saved to: {SCRIPT_PATH}")



[Info] Found 174 jsonl files.
[Info] Sampling every 20th file by order (1-based) -> 8 files selected.
  - Processing step file: 40.jsonl (step=40)
  - Processing step file: 80.jsonl (step=80)
  - Processing step file: 127.jsonl (step=127)
  - Processing step file: 183.jsonl (step=183)
  - Processing step file: 234.jsonl (step=234)


  - Processing step file: 285.jsonl (step=285)
  - Processing step file: 327.jsonl (step=327)
  - Processing step file: 369.jsonl (step=369)
[Done] Processing complete.
Outputs are (per image) under: /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_{step}/image_{image_index}_text_analysis.csv


In [13]:
# -*- coding: utf-8 -*-
"""
Extend per-image CSVs by embedding figures into Excel workbooks.

For each CSV: step_{step}/image_{image_index}_text_analysis.csv
 -> produce:  step_{step}/imagex_{image_index}_full_analysis.xlsx
with three figures per rollout:
  - fig1: original image resized to 256x256
  - fig2: crop region (coords are in 256x256 space; remapped to original, cropped, then resized to 256x256)
  - fig3: fig1 + red rectangle for crop + caption "Disease | pred: <yes/no> | gt: <YES/NO>"

Usage:
  python extend_with_images.py \
    --analysis_root "verl_new/.../rollout_crop_analysis" \
    --full_parquet "/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet"
"""

from pathlib import Path
import argparse
import json
import math
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def map_coords_256_to_orig(coords, orig_w, orig_h):
    """
    coords in 256x256 space -> map to original image space
    coords: [x1, y1, x2, y2]
    """
    if coords is None or len(coords) != 4:
        return None
    x1, y1, x2, y2 = coords
    x1p = int(round(x1 / 256.0 * orig_w))
    x2p = int(round(x2 / 256.0 * orig_w))
    y1p = int(round(y1 / 256.0 * orig_h))
    y2p = int(round(y2 / 256.0 * orig_h))
    # clamp
    x1p = clamp(x1p, 0, orig_w - 1)
    x2p = clamp(x2p, 0, orig_w - 1)
    y1p = clamp(y1p, 0, orig_h - 1)
    y2p = clamp(y2p, 0, orig_h - 1)
    # ensure ordering
    if x2p <= x1p: x2p = min(orig_w - 1, x1p + 1)
    if y2p <= y1p: y2p = min(orig_h - 1, y1p + 1)
    return [x1p, y1p, x2p, y2p]

def map_coords_orig_to_256(coords_orig, orig_w, orig_h):
    """Original-space coords to 256 space."""
    x1, y1, x2, y2 = coords_orig
    x1s = int(round(x1 / float(orig_w) * 256))
    y1s = int(round(y1 / float(orig_h) * 256))
    x2s = int(round(x2 / float(orig_w) * 256))
    y2s = int(round(y2 / float(orig_h) * 256))
    # clamp to [0,255]
    x1s = clamp(x1s, 0, 255); y1s = clamp(y1s, 0, 255)
    x2s = clamp(x2s, 0, 255); y2s = clamp(y2s, 0, 255)
    # ensure order
    if x2s <= x1s: x2s = min(255, x1s + 1)
    if y2s <= y1s: y2s = min(255, y1s + 1)
    return [x1s, y1s, x2s, y2s]

def draw_caption_on_image(img_256: Image.Image, caption: str) -> Image.Image:
    """Draw a semi-transparent caption bar at the bottom with text."""
    img = img_256.convert("RGBA")
    w, h = img.size
    overlay = Image.new("RGBA", (w, h), (0,0,0,0))
    draw = ImageDraw.Draw(overlay)

    # background rectangle
    bar_h = 28
    draw.rectangle([(0, h - bar_h), (w, h)], fill=(0, 0, 0, 140))

    # choose font
    try:
        font = ImageFont.truetype("arial.ttf", 14)
    except:
        font = ImageFont.load_default()

    # text
    margin = 6
    draw.text((margin, h - bar_h + (bar_h - 12) // 2), caption, fill=(255,255,255,255), font=font)
    out = Image.alpha_composite(img, overlay).convert("RGB")
    return out

def ensure_pil_image(path) -> Image.Image:
    return Image.open(path).convert("RGB")

def parse_coords_field(val):
    """Coordinates may be JSON string or list; return list[int] or None."""
    if val is None or val == "":
        return None
    if isinstance(val, list):
        return [int(round(x)) for x in val] if len(val) == 4 else None
    if isinstance(val, str):
        try:
            data = json.loads(val)
            if isinstance(data, list) and len(data) == 4:
                return [int(round(x)) for x in data]
        except Exception:
            pass
    return None

def build_xlsx_for_csv(csv_path: Path, parquet_df: pd.DataFrame):
    # image index from filename
    import re
    m = re.search(r"image_(\-?\d+)_text_analysis\.csv$", csv_path.name)
    if not m:
        print(f"[Skip] Not a target CSV: {csv_path.name}")
        return
    image_index = int(m.group(1))

    # Look up in parquet
    row = parquet_df[parquet_df["seed"] == image_index]
    if row.empty:
        print(f"[Warn] seed={image_index} not found in parquet; skip {csv_path.name}")
        return
    row = row.iloc[0]
    orig_path = row["img_original_path"]
    img256_path = row.get("img_256_path", None)  # optional, we still regenerate fig1 from original
    gt_exists = str(row.get("exists", "")).strip().upper()  # YES/NO

    # Load CSV
    df = pd.read_csv(csv_path)
    # Paths for temporary figures
    fig_dir = csv_path.parent / f"figs_{image_index}"
    safe_mkdir(fig_dir)

    # Prepare workbook
    wb = Workbook()
    ws = wb.active
    ws.title = "analysis"

    # Headers
    text_cols = ["image_index", "crop_coordinates", "reasoning", "answer", "score", "disease", "gt_exists"]
    img_cols = ["fig1_full_256", "fig2_crop_256", "fig3_annotated"]
    headers = text_cols + img_cols
    ws.append(headers)

    # Set column widths
    for i, col in enumerate(headers, start=1):
        if col.startswith("fig"):
            ws.column_dimensions[get_column_letter(i)].width = 45  # wide for images
        else:
            ws.column_dimensions[get_column_letter(i)].width = 28 if col in ("reasoning",) else 16

    # Process each rollout row
    for ridx, rec in df.iterrows():
        coords_256 = parse_coords_field(rec.get("crop_coordinates"))
        answer = str(rec.get("answer", "")).strip()
        disease = str(rec.get("disease", "")).strip()
        try:
            # FIG1: original -> 256x256
            img_orig = ensure_pil_image(orig_path)
            W, H = img_orig.size
            fig1 = img_orig.resize((256, 256))
            fig1_path = fig_dir / f"r{ridx:04d}_fig1_full_256.jpg"
            fig1.save(fig1_path, quality=92)

            # FIG2: crop region (map back to orig, crop, resize 256)
            fig2_path = ""
            coords_orig = None
            if coords_256 is not None:
                coords_orig = map_coords_256_to_orig(coords_256, W, H)
                x1o, y1o, x2o, y2o = coords_orig
                crop = img_orig.crop((x1o, y1o, x2o, y2o))
                fig2 = crop.resize((256, 256))
                fig2_path = fig_dir / f"r{ridx:04d}_fig2_crop_256.jpg"
                fig2.save(fig2_path, quality=92)

            # FIG3: annotate fig1 with rectangle + caption
            fig1_for_draw = fig1.copy()
            if coords_256 is not None:
                # draw rectangle on 256 space figure
                draw = ImageDraw.Draw(fig1_for_draw)
                x1, y1, x2, y2 = [int(v) for v in coords_256]
                # ensure valid box
                x1 = clamp(x1, 0, 255); x2 = clamp(x2, 0, 255)
                y1 = clamp(y1, 0, 255); y2 = clamp(y2, 0, 255)
                if x2 <= x1: x2 = min(255, x1 + 1)
                if y2 <= y1: y2 = min(255, y1 + 1)
                # thickness 3
                for t in range(3):
                    draw.rectangle([x1-t, y1-t, x2+t, y2+t], outline=(255,0,0))

            caption = f"{disease} | pred: {answer} | gt: {gt_exists}"
            fig3 = draw_caption_on_image(fig1_for_draw, caption)
            fig3_path = fig_dir / f"r{ridx:04d}_fig3_annotated.jpg"
            fig3.save(fig3_path, quality=92)

            # Write row text values
            row_vals = [
                rec.get("image_index", image_index),
                json.dumps(coords_256) if coords_256 is not None else "",
                str(rec.get("reasoning", "")),
                answer,
                rec.get("score", ""),
                disease,
                gt_exists,
            ]
            ws.append(row_vals)
            excel_row = ws.max_row

            # Adjust row height to accommodate images (~256 px -> about 190-200 points)
            ws.row_dimensions[excel_row].height = 200

            # Embed images into the row
            # Columns after text are positions len(text_cols)+1 .. +3
            def add_img_to_cell(img_path: Path, col_idx: int):
                if not img_path or str(img_path) == "":
                    return
                try:
                    xi = XLImage(str(img_path))
                    # Set image width/height to 256 to keep consistent
                    # openpyxl will scale based on DPI inside file; we can trust defaults or set explicitly
                    ws.add_image(xi, f"{get_column_letter(col_idx)}{excel_row}")
                except Exception as e:
                    print(f"[Warn] Failed to embed image {img_path}: {e}")

            add_img_to_cell(fig1_path, len(text_cols) + 1)
            add_img_to_cell(fig2_path, len(text_cols) + 2)
            add_img_to_cell(fig3_path, len(text_cols) + 3)

        except Exception as e:
            print(f"[Error] row {ridx} for image {image_index}: {e}")
            # still append text-only row
            row_vals = [
                rec.get("image_index", image_index),
                json.dumps(coords_256) if coords_256 is not None else "",
                str(rec.get("reasoning", "")),
                answer,
                rec.get("score", ""),
                disease,
                gt_exists,
            ]
            ws.append(row_vals)

    # Save Excel next to CSV
    xlsx_path = csv_path.parent / f"imagex_{image_index}_full_analysis.xlsx"
    wb.save(xlsx_path)
    print(f"[OK] Wrote {xlsx_path}")

def run(analysis_root: Path, full_parquet: Path):
    if not analysis_root.exists():
        print(f"[Error] analysis_root not found: {analysis_root}")
        return
    if not full_parquet.exists():
        print(f"[Error] full_parquet not found: {full_parquet}")
        return

    # Load parquet
    try:
        df_full = pd.read_parquet(full_parquet)
    except Exception as e:
        print(f"[Error] reading parquet: {e}")
        return

    # Iterate all step dirs and csvs
    for step_dir in sorted(analysis_root.glob("step_*")):
        for csv_path in sorted(step_dir.glob("image_*_text_analysis.csv")):
            build_xlsx_for_csv(csv_path, df_full)

# if __name__ == "__main__":
#     import sys
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--analysis_root", type=str, required=True)
#     parser.add_argument("--full_parquet", type=str, required=True)
#     args = parser.parse_args()

analysis_root = "/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis"
full_parquet = "/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet"

run(Path(analysis_root), Path(full_parquet))


[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_127/imagex_103720_full_analysis.xlsx
[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_127/imagex_105088_full_analysis.xlsx
[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_127/imagex_113322_full_analysis.xlsx
[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_127/imagex_113723_full_analysis.xlsx
[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis/step_127/imagex_139829_full_analysis.xlsx
[OK] Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.

In [15]:
import shutil, sys, os
src = "/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis"
dst = "/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis.zip"
shutil.make_archive(os.path.splitext(dst)[0], 'zip', src)
print("Wrote", dst)

Wrote /home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis.zip


In [17]:
import zipfile

zip_path = "/home/aiscuser/verl_new/generations/cxr_8k_bal_tool_qwen2.5-vl-7b-instruct_7.0.6_dapo_second_try_full_test/rollout_crop_analysis.zip"
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("包含文件数量:", len(zf.infolist()))
    for name in zf.namelist()[:20]:  # 只预览前20个
        print(name)


包含文件数量: 11227
step_127/
step_183/
step_234/
step_285/
step_327/
step_369/
step_40/
step_80/
step_234/figs_107544/
step_234/figs_117309/
step_234/figs_123306/
step_234/figs_124131/
step_234/figs_12587/
step_234/figs_142523/
step_234/figs_159137/
step_234/figs_164191/
step_234/figs_169486/
step_234/figs_172464/
step_234/figs_174429/
step_234/figs_181865/
